# 3D fermionic toric code — $h_z$ phase-diagram sweep (self-contained)

Matrix-free Krylov diagonalization of the **3D fermionic toric code** at $L=2$
PBC ($N = 3\cdot2^3 = 24$ qubits, $2^{24}\approx16.8$M states).  Everything is
inlined — **no `netket`, no `jax`, no repo imports** — only `numpy`, `scipy`,
`numba`.

The decorated plaquette stabilizer is
$$\tilde B_p = \Big(\prod_{e\in\partial p}\sigma^z_e\Big)\,\sigma^x_{e_+}\,\sigma^x_{e_-},$$
with the two $\sigma^x$ on opposite transverse corner edges (a body diagonal:
the $(+a,+b)$ corner edge on the $+$perp side and the $(-a,-b)$ corner edge on
the $-$perp side).  Vertex stars $A_v$ are the usual all-$\sigma^x$ stars.

**Order parameter.**  The bare $\sigma^z$ Wilson string is *not* conserved (it
anticommutes with the decorated plaquettes), so we **dress it with $\sigma^x$**
(`dressed_string`, a small GF(2) solve) so the operator commutes with every
$\tilde B_p$.  The closed wrapping loop becomes a conserved Wilson loop $W$; the
open half-string *cannot* be made flux-free — each endpoint carries a charge
**and** a flux (the emergent **fermion**) — and string $S$ creates it.  We form
$O_{FM}=\langle S\rangle/\sqrt{|\langle W\rangle|}$.  At $L=2$ the open string
has no bulk, so $O_{FM}$ may be inconclusive; $\langle W\rangle$, the gap, and
$\langle M_z\rangle$ are the reliable signals (a clean $O_{FM}$ needs
$L\ge3$–$4$, past ED reach).

> **Runtime.**  Colab CPU runtime is enough (eigsh is CPU-only).  At $N=24$ a
> single `eigsh` needs ~2.7 GB of Lanczos workspace; the free tier (~12 GB RAM)
> handles it.  A 20-point sweep takes on the order of tens of minutes.

In [ ]:
# Colab already ships numpy/scipy/numba; this is a no-op if they're present.
!pip install -q numba

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from collections import defaultdict
from numba import njit, prange
from scipy.sparse.linalg import LinearOperator, eigsh

## Geometry (netket-free 3D toric code, PBC)

In [ ]:
class ThreeDToricPBC:
    """Minimal 3D toric-code geometry on an L^3 cubic lattice with PBC.

    Qubits live on edges.  An edge along axis d at integer vertex (x,y,z) has
    midpoint (x,y,z)+0.5 e_d; it is keyed by its integer 2x-coordinate tuple
    (avoids float-equality issues).  Exposes the interface the Hamiltonian and
    decoration code consume: N, Lx/Ly/Lz, _coord_to_idx, vertex_all, plaq_all.
    """
    def __init__(self, L: int):
        self.Lx = self.Ly = self.Lz = L
        self.N = 3 * L**3
        self._coord_to_idx = {}
        idx = 0
        for d in range(3):
            for x in range(L):
                for y in range(L):
                    for z in range(L):
                        key = (2*x + (d == 0), 2*y + (d == 1), 2*z + (d == 2))
                        self._coord_to_idx[key] = idx
                        idx += 1
        self.vertex_all = self._stars()
        self.plaq_all = self._plaqs()

    def _idx(self, kx, ky, kz):
        return self._coord_to_idx[(kx % (2*self.Lx), ky % (2*self.Ly), kz % (2*self.Lz))]

    def _stars(self):
        out = []
        for x in range(self.Lx):
            for y in range(self.Ly):
                for z in range(self.Lz):
                    out.append([
                        self._idx(2*x+1, 2*y, 2*z), self._idx(2*x-1, 2*y, 2*z),
                        self._idx(2*x, 2*y+1, 2*z), self._idx(2*x, 2*y-1, 2*z),
                        self._idx(2*x, 2*y, 2*z+1), self._idx(2*x, 2*y, 2*z-1),
                    ])
        return out

    def _plaqs(self):
        out = []
        for c in range(3):                       # normal axis
            a, b = [i for i in range(3) if i != c]
            for x in range(self.Lx):
                for y in range(self.Ly):
                    for z in range(self.Lz):
                        center = [2*x, 2*y, 2*z]
                        center[a] += 1; center[b] += 1
                        edges = []
                        for axis, s in [(a, +1), (a, -1), (b, +1), (b, -1)]:
                            k = list(center); k[axis] += s
                            edges.append(self._idx(*k))
                        out.append(edges)
        return out

## Matrix-free Hamiltonian and Pauli-string expectations

In [ ]:
@njit(inline='always')
def _bit_parity64(x):
    x ^= x >> 32; x ^= x >> 16; x ^= x >> 8
    x ^= x >> 4;  x ^= x >> 2;  x ^= x >> 1
    return x & 1


@njit(parallel=True, fastmath=True)
def _matvec_jit(psi, diag, x_masks, x_coefs, out):
    dim = psi.shape[0]; n_masks = x_masks.shape[0]
    for i in prange(dim):
        s = diag[i] * psi[i]
        for k in range(n_masks):
            s += x_coefs[k] * psi[i ^ x_masks[k]]
        out[i] = s


@njit(parallel=True, fastmath=True)
def _matvec_jit_xz_add(psi, xz_z_masks, xz_x_masks, xz_coefs, out):
    dim = psi.shape[0]; n = xz_z_masks.shape[0]
    for j in prange(dim):
        s = 0.0
        for k in range(n):
            sign = 1.0 - 2.0 * _bit_parity64(j & xz_z_masks[k])
            s += xz_coefs[k] * sign * psi[j ^ xz_x_masks[k]]
        out[j] += s


def qubits_to_mask(indices):
    m = 0
    for i in indices:
        if i == -1:
            continue
        m |= (1 << int(i))
    return m


def z_string_eigvals(basis, mask, N):
    parity = np.zeros_like(basis, dtype=np.int8)
    for i in range(N):
        if mask & (1 << i):
            parity ^= ((basis >> i) & 1).astype(np.int8)
    return 1.0 - 2.0 * parity.astype(np.float64)


def hamiltonian_linop(geom, hx=0.0, hy=0.0, hz=0.0, J=1.0, xz_stabs=None, dtype=np.float64):
    """Matrix-free toric-code Hamiltonian -> (LinearOperator, basis).

    xz_stabs: optional list of (z_qubits, x_qubits, coef) generalized stabilizers.
    When non-empty they REPLACE the default Z-only plaquette term.
    """
    if hy != 0:
        raise NotImplementedError("Y-field not supported (h_y = 0 assumed).")
    N = geom.N
    dim = 1 << N
    basis = np.arange(dim, dtype=np.int64)
    use_xz = xz_stabs is not None and len(xz_stabs) > 0

    diag = np.zeros(dim, dtype=dtype)
    if not use_xz:
        for p in geom.plaq_all:
            diag -= J * z_string_eigvals(basis, qubits_to_mask(p), N)
    if hz != 0.0:
        for i in range(N):
            diag -= hz * z_string_eigvals(basis, 1 << i, N)

    x_strings = defaultdict(float)
    for v in geom.vertex_all:
        x_strings[qubits_to_mask(v)] -= J
    if hx != 0.0:
        for i in range(N):
            x_strings[1 << i] -= hx

    xz_z_list, xz_x_list, xz_c_list = [], [], []
    if use_xz:
        for z_qubits, x_qubits, coef in xz_stabs:
            zm = qubits_to_mask(z_qubits); xm = qubits_to_mask(x_qubits)
            if zm & xm:
                raise ValueError(f"overlapping z and x supports: z={z_qubits}, x={x_qubits}")
            if xm == 0:
                diag += coef * z_string_eigvals(basis, zm, N)
            elif zm == 0:
                x_strings[xm] += coef
            else:
                xz_z_list.append(zm); xz_x_list.append(xm); xz_c_list.append(coef)

    x_strings = {m: c for m, c in x_strings.items() if c != 0.0 and m != 0}
    if x_strings:
        x_masks = np.fromiter(x_strings.keys(), dtype=np.int64, count=len(x_strings))
        x_coefs = np.fromiter(x_strings.values(), dtype=dtype, count=len(x_strings))
    else:
        x_masks = np.empty(0, dtype=np.int64); x_coefs = np.empty(0, dtype=dtype)

    xz_z_masks = np.array(xz_z_list, dtype=np.int64) if xz_z_list else np.empty(0, dtype=np.int64)
    xz_x_masks = np.array(xz_x_list, dtype=np.int64) if xz_x_list else np.empty(0, dtype=np.int64)
    xz_coefs   = np.array(xz_c_list, dtype=dtype)    if xz_c_list else np.empty(0, dtype=dtype)
    has_xz = xz_z_masks.size > 0

    def matvec(psi):
        psi = np.ascontiguousarray(psi, dtype=dtype)
        out = np.empty_like(psi)
        _matvec_jit(psi, diag, x_masks, x_coefs, out)
        if has_xz:
            _matvec_jit_xz_add(psi, xz_z_masks, xz_x_masks, xz_coefs, out)
        return out

    return LinearOperator((dim, dim), matvec=matvec, dtype=dtype), basis


def expect_x_string(psi, basis, mask):
    if mask == 0:
        return float(np.real(np.sum(np.abs(psi) ** 2)))
    return float(np.real(np.sum(np.conj(psi) * psi[basis ^ mask])))


def expect_z_string(psi, basis, mask, N):
    if mask == 0:
        return float(np.sum(np.abs(psi) ** 2))
    return float(np.sum(np.abs(psi) ** 2 * z_string_eigvals(basis, mask, N)))


def expect_xz_string(psi, basis, z_mask, x_mask, N):
    if x_mask == 0:
        return expect_z_string(psi, basis, z_mask, N)
    if z_mask == 0:
        return expect_x_string(psi, basis, x_mask)
    signs = z_string_eigvals(basis, z_mask, N)
    return float(np.real(np.sum(np.conj(psi) * signs * psi[basis ^ x_mask])))

## Fermionic plaquette decoration

In [ ]:
_E = np.eye(3)


def _idx(geom, coord):
    """Qubit index at edge-midpoint `coord`, PBC-wrapped (2x-integer keys)."""
    L = (geom.Lx, geom.Ly, geom.Lz)
    key = tuple(int(round(2 * coord[d])) % (2 * L[d]) for d in range(3))
    return int(geom._coord_to_idx[key])


def fermionic_plaquettes(geom, J=1.0):
    """Decorated plaquette stabilizers as (z_edges, x_edges, coef) triples.
    Per plaquette: 4 sigma^z boundary edges + two sigma^x corner edges at
    ctr +/- 0.5*(e_a + e_b + e_c) (a body diagonal)."""
    out = []
    for c in range(3):                                   # plaquette normal axis
        a, b = (d for d in range(3) if d != c)           # in-plane axes
        for ix in range(geom.Lx):
            for iy in range(geom.Ly):
                for iz in range(geom.Lz):
                    ctr = np.array([ix, iy, iz], float) + 0.5 * _E[a] + 0.5 * _E[b]
                    z_edges = [_idx(geom, ctr + s * 0.5 * _E[ax])
                               for ax in (a, b) for s in (+1, -1)]
                    diag = 0.5 * (_E[a] + _E[b] + _E[c])
                    x_edges = [_idx(geom, ctr + diag), _idx(geom, ctr - diag)]
                    out.append((z_edges, x_edges, -float(J)))
    return out


def verify_xz_commutation(stabs, vertex_all):
    """Pairwise commutation of (decorated plaquettes) + (vertex stars)."""
    entries = [(qubits_to_mask(z), qubits_to_mask(x)) for z, x, _ in stabs]
    entries += [(0, qubits_to_mask(v)) for v in vertex_all]
    viol = []
    n = len(entries)
    for i in range(n):
        z1, x1 = entries[i]
        for j in range(i + 1, n):
            z2, x2 = entries[j]
            if (bin(z1 & x2).count("1") + bin(z2 & x1).count("1")) & 1:
                viol.append((i, j))
    return {"ok": not viol, "violations": viol, "n_stabilizers": n}


def _gf2_solve(rows, targets, ncols):
    """Best-effort GF(2) solve of  popcount(rows[i] & s) == targets[i]  -> s (bitmask).
    Returns a particular solution of the consistent subsystem (free vars = 0)."""
    R = len(rows)
    aug = [rows[i] | (int(targets[i]) << ncols) for i in range(R)]
    pivot_of = {}; r = 0
    for col in range(ncols):
        sel = next((k for k in range(r, R) if (aug[k] >> col) & 1), None)
        if sel is None:
            continue
        aug[r], aug[sel] = aug[sel], aug[r]
        for k in range(R):
            if k != r and (aug[k] >> col) & 1:
                aug[k] ^= aug[r]
        pivot_of[col] = r; r += 1
    s = 0
    for col, row in pivot_of.items():
        if (aug[row] >> ncols) & 1:
            s |= 1 << col
    return s


def dressed_string(geom, stabs, z_edges):
    """Dress a sigma^z string with sigma^x so Z(z_edges)*X(s) commutes with every
    decorated plaquette.  Returns (z_edges, x_edges, flux_plaqs): flux_plaqs is empty
    for a closed loop (conserved) and lists the endpoint plaquettes for an open string
    (the unavoidable flux of the charge+flux fermion)."""
    zmask = qubits_to_mask(z_edges)
    rows = [qubits_to_mask(z) for z, x, _ in stabs]
    targets = [bin(zmask & qubits_to_mask(x)).count("1") & 1 for z, x, _ in stabs]
    s = _gf2_solve(rows, targets, geom.N)
    assert not (zmask & s), "dressing overlaps the sigma^z line (would give sigma^y)"
    x_edges = [i for i in range(geom.N) if (s >> i) & 1]
    flux_plaqs = [pi for pi, (z, x, _) in enumerate(stabs)
                  if (bin(zmask & qubits_to_mask(x)).count("1")
                      + bin(s & qubits_to_mask(z)).count("1")) & 1]
    return z_edges, x_edges, flux_plaqs

## Setup + zero-field sanity check

In [ ]:
def z_line_qubits(geom, x=0, y=0):
    """sigma^z edges along z at column (x, y), wrapping the torus (PBC)."""
    Lx2, Ly2, Lz2 = 2*geom.Lx, 2*geom.Ly, 2*geom.Lz
    return [geom._coord_to_idx[((2*x) % Lx2, (2*y) % Ly2, (2*z + 1) % Lz2)]
            for z in range(geom.Lz)]


geom = ThreeDToricPBC(2)
stabs_f = fermionic_plaquettes(geom, J=1.0)
chk = verify_xz_commutation(stabs_f, geom.vertex_all)
print(f"fermionic 3D TC L=2 PBC:  N = {geom.N},  |A_v| = {len(geom.vertex_all)},  "
      f"|B~_p| = {len(stabs_f)},  commute = {chk['ok']}  ({len(chk['violations'])} violations)")

# Zero-field sanity: E0 = -(N_v + N_p), every decorated stabilizer +1.
H, basis = hamiltonian_linop(geom, hx=0.0, hz=0.0, J=1.0, xz_stabs=stabs_f)
ev, vc = eigsh(H, k=4, which="SA", tol=1e-7)
psi = vc[:, 0]
A = np.mean([expect_x_string(psi, basis, qubits_to_mask(v)) for v in geom.vertex_all])
B = np.mean([expect_xz_string(psi, basis, qubits_to_mask(z), qubits_to_mask(x), geom.N)
             for z, x, _ in stabs_f])
print(f"eigenvalues: {ev}")
print(f"mean <A_v>  = {A:.4f}    (expect 1.0)")
print(f"mean <B~_p> = {B:.4f}    (expect 1.0)")
print(f"E0 = {ev[0]:.4f}        (expect -{len(geom.vertex_all) + len(stabs_f)})")

# sigma^z strings dressed with sigma^x so they commute with the decorated plaquettes.
cz = z_line_qubits(geom, 0, 0); oz = cz[: max(1, len(cz) // 2)]
W_z, W_x, W_flux = dressed_string(geom, stabs_f, cz)   # conserved closed loop
S_z, S_x, S_flux = dressed_string(geom, stabs_f, oz)   # open fermion string
print(f"closed loop : |s^z|={len(W_z)}, |s^x dressing|={len(W_x)}, "
      f"flux endpoints={len(W_flux)}  (0 => conserved)")
print(f"open string : |s^z|={len(S_z)}, |s^x dressing|={len(S_x)}, "
      f"flux endpoints={len(S_flux)}  (=> charge+flux fermion at each end)")

## $h_z$ sweep

In [ ]:
def sweep_phase_diagram_3d_fermionic(geom, stabs, h_z_list, hx=0.3, k=2,
                                     label="3D fermionic L=2 PBC"):
    """Order parameter: dressed conserved Wilson loop W and open fermion string S,
    combined into the Fredenhagen-Marcu ratio O_FM = <S>/sqrt(|<W>|).  Gap and
    <M_z> are kept as robust diagnostics."""
    N = geom.N
    basis = np.arange(1 << N, dtype=np.int64)
    cz = z_line_qubits(geom, 0, 0); oz = cz[: max(1, len(cz) // 2)]
    W_z, W_x, _ = dressed_string(geom, stabs, cz)
    S_z, S_x, _ = dressed_string(geom, stabs, oz)
    Wz_mask, Wx_mask = qubits_to_mask(W_z), qubits_to_mask(W_x)
    Sz_mask, Sx_mask = qubits_to_mask(S_z), qubits_to_mask(S_x)
    star_masks = [qubits_to_mask(v) for v in geom.vertex_all]
    plaq_zx = [(qubits_to_mask(z), qubits_to_mask(x)) for z, x, _ in stabs]
    site_masks = [1 << i for i in range(N)]

    keys = ("h_z", "E0", "gap", "A", "B", "Mz", "W", "S", "O_FM")
    rec = {kk: [] for kk in keys}
    for h_z in tqdm(h_z_list, desc=label):
        H, _ = hamiltonian_linop(geom, hx=hx, hz=float(h_z), J=1.0, xz_stabs=stabs)
        ev, vc = eigsh(H, k=k, which="SA", tol=1e-7, return_eigenvectors=True)
        psi = vc[:, 0]
        W = expect_xz_string(psi, basis, Wz_mask, Wx_mask, N)   # conserved Wilson loop
        S = expect_xz_string(psi, basis, Sz_mask, Sx_mask, N)   # fermion string
        rec["h_z"].append(float(h_z)); rec["E0"].append(float(ev[0]))
        rec["gap"].append(float(ev[1] - ev[0]))
        rec["A"].append(float(np.mean([expect_x_string(psi, basis, m) for m in star_masks])))
        rec["B"].append(float(np.mean([expect_xz_string(psi, basis, zm, xm, N) for zm, xm in plaq_zx])))
        rec["Mz"].append(float(np.mean([expect_z_string(psi, basis, m, N) for m in site_masks])))
        rec["W"].append(W); rec["S"].append(S)
        rec["O_FM"].append(S / np.sqrt(abs(W)) if abs(W) > 1e-14 else np.nan)
    return {kk: np.array(v) for kk, v in rec.items()}


h_z_list = np.linspace(0, 0.75, 20)
rec_3df = sweep_phase_diagram_3d_fermionic(geom, stabs_f, h_z_list, hx=0.3)

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rec_3df["h_z"], rec_3df["O_FM"], "o-", color="C3", label="3D fermionic L=2 PBC")
axes[1].plot(rec_3df["h_z"], np.gradient(rec_3df["O_FM"], rec_3df["h_z"]),
             "o-", color="C3", label="3D fermionic L=2 PBC")
axes[0].set_ylabel(r"$O_{FM} = \langle S\rangle/\sqrt{|\langle W\rangle|}$")
axes[1].set_ylabel(r"$\partial O_{FM}/\partial h_z$")
for ax in axes:
    ax.set_xlabel(r"$h_z$"); ax.legend()
fig.suptitle(r"3D fermionic TC, $h_x = 0.3$: dressed Fredenhagen--Marcu ratio")
fig.tight_layout(); plt.show()

# Conserved Wilson loop and spectral gap.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rec_3df["h_z"], rec_3df["W"], "o-", color="C3")
axes[1].plot(rec_3df["h_z"], rec_3df["gap"], "o-", color="C3")
axes[0].set_ylabel(r"$\langle W\rangle$ (conserved loop)")
axes[1].set_ylabel(r"spectral gap")
for ax in axes:
    ax.set_xlabel(r"$h_z$")
fig.suptitle(r"3D fermionic TC, $h_x = 0.3$: conserved loop and gap")
fig.tight_layout(); plt.show()


def plot_obs_and_derivs(rec, suptitle, color="C3", b_label=r"$\langle \tilde B_p\rangle$"):
    """Top row: <A_v>, <B_p>, <M_z> expectations;  bottom row: their h_z-derivatives."""
    cols = [("A", r"$\langle A_v\rangle$"), ("B", b_label), ("Mz", r"$\langle M_z\rangle$")]
    h = rec["h_z"]
    fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
    for j, (key, lab) in enumerate(cols):
        axes[0, j].plot(h, rec[key], "o-", color=color)
        axes[0, j].set_ylabel(lab)
        axes[1, j].plot(h, np.gradient(rec[key], h), "o-", color=color)
        axes[1, j].set_ylabel(r"$\partial$ " + lab + r"$ / \partial h_z$")
    for ax in axes[1, :]:
        ax.set_xlabel(r"$h_z$")
    fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()


plot_obs_and_derivs(rec_3df, r"3D fermionic TC L=2 PBC: stabilizer & magnetization expectations")